# SoundCluster — Segmentación de Usuarios para Personalización en Streaming Musical
**Caso de estudio 3 · EG-S-IA-03-26 · Período 1/2026**

Este notebook implementa la solución completa siguiendo la metodología **CRISP-DM**
(Cross-Industry Standard Process for Data Mining), la misma que estructura el proyecto.

**Problema:** clustering / segmentación **no supervisada**. No hay etiqueta correcta que
predecir; el objetivo es descubrir segmentos naturales de usuarios por sus hábitos de escucha.

**Modelo principal:** K-Means (sobre dataset sintético).
**Track paralelo:** K-Prototypes sobre la encuesta real de Kaggle (demostración metodológica).

> Estructura del notebook = las 6 fases de CRISP-DM + un track paralelo al final.


## 0. Configuración e instalación

In [ ]:
# En Colab, descomenta para instalar las dependencias que no vienen por defecto.
# !pip install -q yellowbrick kmodes plotly


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Semilla global para reproducibilidad (buena práctica CRISP-DM)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estilo de gráficos
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 110

print("Entorno listo. NumPy", np.__version__, "| Pandas", pd.__version__)


Entorno listo. NumPy 2.4.6 | Pandas 3.0.3


## 1. CRISP-DM · Fase 1 — Comprensión del negocio

**Contexto.** SoundFlow es una plataforma latinoamericana de streaming con 15M de usuarios
activos. Sus campañas y recomendaciones son genéricas y no conectan con la diversidad de
hábitos de escucha. Quieren pasar de recomendar canciones sueltas a **comprender perfiles**
de usuario para diseñar experiencias a medida (playlists dinámicas, paquetes de suscripción
diferenciados, promociones segmentadas por ciudad).

**Objetivo del negocio.** Identificar segmentos de usuarios con patrones de escucha
homogéneos para personalizar experiencia y marketing.

**Objetivo de minería de datos.** Aplicar clustering (K-Means) sobre variables de
comportamiento para descubrir esos segmentos y perfilarlos.

**Criterios de éxito.** Segmentos internamente cohesionados y bien separados
(silueta alta, Davies-Bouldin baja), interpretables, y accionables como estrategias concretas.


## 2. CRISP-DM · Fase 2 — Comprensión de los datos

Trabajamos con **dos fuentes en paralelo**:

- **Principal (sintético):** ~10.000 usuarios con variables numéricas de comportamiento.
  Lo generamos nosotros con *semillas* (perfiles latentes) + *ruido* + *solape* realista,
  de modo que los clusters existen pero **no están regalados**: K-Means debe descubrirlos.
- **Secundario (encuesta real de Kaggle):** para el track K-Prototypes (ver sección final).

> **Nota de honestidad metodológica (clave para la defensa).** Guardamos la etiqueta real del
> segmento en una columna aparte (`segmento_real`) que **NO** entra al clustering; la usamos
> solo al final para validar con el Adjusted Rand Index (ARI) cuánto se parecen los clusters
> descubiertos a los latentes. Como el dataset es sintético y tiene estructura plantada, el ARI
> saldrá alto (~0.9): eso **confirma que el pipeline no supervisado recupera correctamente la
> estructura**, no que los datos estén "regalados". Los perfiles se calibraron con solape real
> (silueta ~0.28 en el K óptimo, un valor realista) para que la elección de K sea un resultado
> genuino y no un artefacto. Se declara explícitamente que los datos son sintéticos.


### 2.1 Generación del dataset sintético

In [5]:
from dataclasses import dataclass


@dataclass
class PerfilLatente:
    """Perfil medio de un segmento latente. Las mezclas de género son proporciones
    (rock, pop, electronica, clasica) que suman ~1 y luego se pasan a porcentaje."""
    nombre: str
    peso: float                      # proporción de usuarios de este segmento
    horas_semana: float              # media de horas de escucha semanales
    mezcla_genero: tuple             # (rock, pop, electronica, clasica) ~ suma 1
    pct_artistas_nuevos: float       # % de escucha dedicada a artistas nuevos
    pct_playlists_propias: float     # % de escucha desde playlists propias
    horario: tuple                   # probabilidades (mañana, tarde, noche)
    me_gusta_semana: float           # media de "me gusta" por semana
    pct_saltadas: float              # % de canciones saltadas
    playlists_creadas: float         # media de playlists creadas


# Cinco perfiles latentes CALIBRADOS con solape deliberado: los valores escalares se
# comprimieron hacia la media global (factor 0.6) para que los segmentos compartan terreno
# y K-Means tenga que trabajar de verdad, en lugar de recibir clusters regalados.
PERFILES = [
    PerfilLatente("Descubridores nocturnos", 0.20, 25.0, (0.20, 0.15, 0.45, 0.20),
                  34.8, 48.2, (0.10, 0.25, 0.65), 26, 33.9, 11),
    PerfilLatente("Clasicos de fin de semana", 0.18, 16.6, (0.10, 0.15, 0.05, 0.70),
                  12.6, 33.2, (0.20, 0.55, 0.25), 14, 15.9, 5),
    PerfilLatente("Usuario de fondo", 0.24, 29.8, (0.25, 0.45, 0.20, 0.10),
                  11.4, 24.2, (0.45, 0.40, 0.15), 10, 44.1, 4),
    PerfilLatente("Fans del pop diurno", 0.22, 20.2, (0.15, 0.60, 0.15, 0.10),
                  19.8, 39.2, (0.50, 0.40, 0.10), 23, 23.1, 8),
    PerfilLatente("Rockeros intensos", 0.16, 26.2, (0.65, 0.15, 0.15, 0.05),
                  18.6, 45.2, (0.30, 0.45, 0.25), 25, 21.9, 9),
]


def generar_dataset_sintetico(n_usuarios: int = 10_000,
                              seed: int = RANDOM_STATE,
                              incluir_etiqueta: bool = True) -> pd.DataFrame:
    """Genera un dataset sintético de comportamiento de usuarios de streaming.

    Los clusters son identificables pero se solapan (ruido gaussiano + concentración
    Dirichlet moderada en la mezcla de géneros), así K-Means tiene que trabajar.

    Args:
        n_usuarios: número de filas (usuarios).
        seed: semilla para reproducibilidad.
        incluir_etiqueta: si True, agrega la columna `segmento_real` (NO usar en clustering).

    Returns:
        DataFrame con las variables de comportamiento (y opcionalmente la etiqueta real).
    """
    rng = np.random.default_rng(seed)
    pesos = np.array([p.peso for p in PERFILES])
    pesos = pesos / pesos.sum()
    asignacion = rng.choice(len(PERFILES), size=n_usuarios, p=pesos)

    filas = []
    for idx in asignacion:
        p = PERFILES[idx]

        # Horas de escucha: normal truncada a [1, 60]
        horas = np.clip(rng.normal(p.horas_semana, 6), 1, 60)

        # Mezcla de géneros: Dirichlet centrada en la mezcla del perfil (conc=14 -> solape)
        mezcla = rng.dirichlet(np.array(p.mezcla_genero) * 14 + 0.5)
        pct_rock, pct_pop, pct_elec, pct_clas = (mezcla * 100)

        # Horario predominante: 0=mañana, 1=tarde, 2=noche
        horario = rng.choice([0, 1, 2], p=p.horario)

        fila = {
            "horas_semana": round(horas, 1),
            "pct_rock": round(pct_rock, 1),
            "pct_pop": round(pct_pop, 1),
            "pct_electronica": round(pct_elec, 1),
            "pct_clasica": round(pct_clas, 1),
            "pct_artistas_nuevos": round(np.clip(rng.normal(p.pct_artistas_nuevos, 10), 0, 100), 1),
            "pct_playlists_propias": round(np.clip(rng.normal(p.pct_playlists_propias, 12), 0, 100), 1),
            "horario_predominante": int(horario),
            "me_gusta_semana": int(np.clip(rng.normal(p.me_gusta_semana, 7), 0, None)),
            "pct_saltadas": round(np.clip(rng.normal(p.pct_saltadas, 9), 0, 100), 1),
            "playlists_creadas": int(np.clip(rng.normal(p.playlists_creadas, 4), 0, None)),
        }
        if incluir_etiqueta:
            fila["segmento_real"] = p.nombre
        filas.append(fila)

    df = pd.DataFrame(filas)

    # Inyectamos ~1.5% de valores nulos en dos columnas, para practicar imputación en Fase 3.
    for col in ["me_gusta_semana", "pct_saltadas"]:
        mask = rng.random(len(df)) < 0.015
        df.loc[mask, col] = np.nan

    return df


df = generar_dataset_sintetico(n_usuarios=10_000)
print("Dimensiones:", df.shape)
df.head()


Dimensiones: (10000, 12)


,horas_semana,pct_rock,pct_pop,pct_electronica,pct_clasica,pct_artistas_nuevos,pct_playlists_propias,horario_predominante,me_gusta_semana,pct_saltadas,playlists_creadas,segmento_real
0,8.9,31.3,51.2,13.8,3.7,1.5,37.7,0,21.0,25.6,5,Fans del pop diurno
1,41.1,22.5,39.4,23.0,15.1,10.6,21.8,0,14.0,44.3,6,Usuario de fondo
2,18.7,57.3,18.0,24.1,0.6,20.9,25.5,1,20.0,8.4,2,Rockeros intensos
3,20.9,19.4,62.9,11.3,6.5,20.7,37.8,0,44.0,11.0,7,Fans del pop diurno
4,16.5,17.7,28.6,45.1,8.6,35.5,54.9,2,24.0,31.6,4,Descubridores nocturnos


In [6]:
# Vista general
df.info()
df.describe().round(1)


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   horas_semana           10000 non-null  float64
 1   pct_rock               10000 non-null  float64
 2   pct_pop                10000 non-null  float64
 3   pct_electronica        10000 non-null  float64
 4   pct_clasica            10000 non-null  float64
 5   pct_artistas_nuevos    10000 non-null  float64
 6   pct_playlists_propias  10000 non-null  float64
 7   horario_predominante   10000 non-null  int64  
 8   me_gusta_semana        9851 non-null   float64
 9   pct_saltadas           9849 non-null   float64
 10  playlists_creadas      10000 non-null  int64  
 11  segmento_real          10000 non-null  str    
dtypes: float64(9), int64(2), str(1)
memory usage: 937.6 KB


,horas_semana,pct_rock,pct_pop,pct_electronica,pct_clasica,pct_artistas_nuevos,pct_playlists_propias,horario_predominante,me_gusta_semana,pct_saltadas,playlists_creadas
count,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,9851.0,9849.0,10000.0
mean,23.9,25.2,31.0,21.3,22.5,19.5,37.3,1.0,18.8,29.1,7.0
std,7.6,18.4,19.7,15.1,22.0,12.6,14.8,0.8,9.4,13.6,4.5
min,1.0,0.2,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,18.6,12.0,13.8,9.7,7.1,9.9,27.5,0.0,12.0,19.1,4.0
50%,24.0,19.9,26.8,17.7,14.1,18.5,37.4,1.0,19.0,28.3,7.0
75%,29.1,32.4,46.7,29.9,27.6,28.0,47.6,2.0,26.0,38.9,10.0
max,47.5,93.4,86.7,79.6,92.8,69.9,90.0,2.0,55.0,73.0,25.0


In [7]:
# Distribución de la etiqueta latente (solo referencia; NO se usa en el clustering)
df["segmento_real"].value_counts()


segmento_real
Usuario de fondo             2416
Fans del pop diurno          2182
Descubridores nocturnos      2034
Clasicos de fin de semana    1800
Rockeros intensos            1568
Name: count, dtype: int64

### 2.2 Track paralelo — Carga de la encuesta real (Kaggle)

Dataset: *Spotify User Behavior Dataset* (meeraajayakumar). Es una **encuesta** con variables
casi todas categóricas → por eso su track usa **K-Prototypes**, no K-Means (ver sección 7).


In [8]:
# Descarga manual desde Kaggle y sube el CSV a Colab, luego:
# encuesta = pd.read_csv("spotify_user_behavior.csv")
# encuesta.head()
# TODO(Fase 2): EDA de la encuesta -> tipos, nulos, cardinalidad de cada categórica.


## 3. CRISP-DM · Fase 3 — Preparación de los datos

Pasos obligatorios para K-Means (sensible a escala y a outliers):
1. **Nulos** → imputación por mediana (o eliminación justificada).
2. **Outliers** → detección con IQR (o Z-score) y tratamiento (recorte/winsorizing).
3. **Escalado** → `StandardScaler` (obligatorio). Guardar el scaler para el despliegue.


In [9]:
# TODO(3.1) Imputación de nulos por mediana
# from sklearn.impute import SimpleImputer

# TODO(3.2) Detección/tratamiento de outliers con IQR
# def limitar_outliers_iqr(serie, k=1.5): ...

# TODO(3.3) Escalado con StandardScaler (ajustar SOLO sobre X, sin la etiqueta)
# from sklearn.preprocessing import StandardScaler
# X = df.drop(columns=["segmento_real"])
# scaler = StandardScaler().fit(X)
# X_scaled = scaler.transform(X)


## 4. CRISP-DM · Fase 4 — Modelado

1. **Número óptimo de K:** método del codo (inercia) + coeficiente de silueta, K de 2 a 10.
2. **Entrenar K-Means** con el K elegido.
3. **PCA** a 2D para visualizar los clusters coloreados (t-SNE opcional).


In [10]:
# TODO(4.1) Método del codo + silueta para K en 2..10
# from sklearn.cluster import KMeans
# from sklearn.metrics import silhouette_score

# TODO(4.2) Entrenar KMeans(n_clusters=K_opt, random_state=RANDOM_STATE, n_init=10)

# TODO(4.3) PCA(n_components=2) para el scatter coloreado por cluster
# from sklearn.decomposition import PCA


## 5. CRISP-DM · Fase 5 — Evaluación e interpretación

- **Validación interna:** Davies-Bouldin (menor es mejor) y Calinski-Harabasz (mayor es mejor).
- **Perfiles:** tabla de promedios por cluster + radar charts / boxplots comparativos.
- **Nombre y narrativa** por cluster ("Los Descubridores Nocturnos", etc.).
- **Estabilidad:** bootstrapping o partición de la muestra (consistencia de los clusters).
- **Validación externa (bonus del sintético):** Adjusted Rand Index vs. `segmento_real`.


In [11]:
# TODO(5.1) davies_bouldin_score / calinski_harabasz_score
# TODO(5.2) df.groupby("cluster").mean()  -> tabla de perfiles
# TODO(5.3) radar charts / boxplots por cluster
# TODO(5.4) estabilidad por bootstrapping (re-clustering en submuestras)
# TODO(5.5) adjusted_rand_score(df["segmento_real"], df["cluster"])


## 6. CRISP-DM · Fase 6 — Despliegue

- Serializar `scaler.pkl` y `kmeans_soundcluster.pkl` con joblib.
- Construir la tabla de estrategias por segmento (playlists, oferta, horario de notificación).
- Estos artefactos los consumirá la API FastAPI del prototipo (paso 5 del plan).


In [12]:
# import joblib
# joblib.dump(scaler, "models/scaler.pkl")
# joblib.dump(kmeans, "models/kmeans_soundcluster.pkl")
# TODO(6): función asignar_segmento(perfil_usuario) -> nombre_segmento + estrategia


## 7. Track paralelo — K-Prototypes sobre la encuesta real

Demostración de criterio: K-Means no aplica a datos casi todo categóricos.
- Encoding mínimo (mantener numéricas como `Age` si existen).
- `KPrototypes` de la librería `kmodes`, indicando índices de columnas categóricas.
- Comparar cohesión/interpretabilidad con el resultado del sintético.


In [13]:
# from kmodes.kprototypes import KPrototypes
# TODO(7): definir columnas categóricas, elegir K por costo, ajustar y perfilar.
